# Getting Started with Hugging Face Pipelines

The Hugging Face `transformers` library is the most popular way to use open-source AI models.

The easiest way to start is using **Pipelines**. They handle all the complex logic (tokenization, model loading, and post-processing) for you in just a few lines of code.

### The Basic Pattern:
1. **Initialize**: `my_pipeline = pipeline("task-name")`
2. **Run**: `result = my_pipeline("your input text")`

## Key Concept: Inference vs. Training

When working with AI models, there are two main phases:

### 1. Training (and Fine-Tuning)
**Training** is the process of building a model from scratch or teaching it new things using a dataset. This updates the model's internal 'knowledge' (weights). If we take a pre-trained model and train it just a little bit more on specific data, we call it **Fine-Tuning**.

### 2. Inference
**Inference** is simply using a model that is already trained to get an answer. When you give a prompt to Gemini or ChatGPT and get a response, you are performing inference.

**The Pipelines API is designed for fast and easy Inference.**

In [ ]:
# Step 1: Install the necessary libraries
!pip install -q --upgrade datasets transformers diffusers accelerate

In [ ]:
# Step 2: Verify GPU availability
# Using a GPU (like the Tesla T4 in Colab) makes model inference much faster.
import torch
if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU. Models might be slower.")

In [ ]:
# Imports
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio

In [ ]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

## Using Pipelines from Hugging Face

A simple way to run inference for common tasks, without worrying about all the plumbing, picking reasonable defaults.


### How it works:

STEP 1: Create a pipeline - a function you can then call

```python
my_pipeline = pipeline(task, model=xx, device=xx)
```

If you don't specify a model, then Hugging Face picks one for you that's the default for the task. Specify "cuda" for the device to use an NVIDIA GPU like the one on the T4. Specify "mps" on a Mac.


STEP 2: Then call it as many times as you want:

```python
my_pipeline(input1)
my_pipeline(input2)
```

In [ ]:
# Example 1: Sentiment Analysis
# This automatically detects if a text is positive or negative.
classifier = pipeline("sentiment-analysis", device="cuda" if torch.cuda.is_available() else "cpu")
result = classifier("I am learning how to use Hugging Face and it's amazing!")
print(result)

In [ ]:
result = my_simple_sentiment_analyzer("I should be more excited to be on the way to LLM mastery!")
print(result)

In [ ]:
better_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda")
result = better_sentiment("I should be more excited to be on the way to LLM mastery!!")
print(result)

In [ ]:
# Example 2: Named Entity Recognition (NER)
# This identifies real-world objects like people, places, and organizations.
ner_pipe = pipeline("ner", device="cuda" if torch.cuda.is_available() else "cpu")
text = "Hugging Face is based in New York and was founded by Clement Delangue."
for entity in ner_pipe(text):
    print(entity)

In [ ]:
# Example 3: Question Answering
# Provide a 'context' (knowledge) and ask a question based on it.
qa_pipe = pipeline("question-answering", device="cuda" if torch.cuda.is_available() else "cpu")
context = "Pipelines are the simplest way to use models for inference."
question = "What are pipelines used for?"

result = qa_pipe(question=question, context=context)
print(f"Answer: {result['answer']}")

In [ ]:
# Text Summarization
summarizer = pipeline("summarization", device="cuda")
text = """
The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.
"""

summary = summarizer(text, max_length=50, min_length=25, do_sample=False)
print(summary[0]['summary_text'])

In [ ]:
# Example 4: Translation
# You can specify the direction of translation directly in the task name.
translator = pipeline("translation_en_to_fr", device="cuda" if torch.cuda.is_available() else "cpu")
result = translator("Learning AI is a journey, not a destination.")
print(result[0]['translation_text'])

In [ ]:
# Another translation, showing a model being specified
# All translation models are here: https://huggingface.co/models?pipeline_tag=translation&sort=trending
translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es", device="cuda")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])

In [ ]:
# Classification
classifier = pipeline("zero-shot-classification", device="cuda")
result = classifier("Hugging Face's Transformers library is amazing!", candidate_labels=["technology", "sports", "politics"])
print(result)

In [ ]:
# Text Generation
generator = pipeline("text-generation", device="cuda")
result = generator("If there's one thing I want you to remember about using HuggingFace pipelines, it's")
print(result[0]['generated_text'])

In [ ]:
# Image Generation - remember this?! Now you know what's going on
# Pipelines can be used for diffusion models as well as transformers
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)

In [ ]:
# Audio Generation
from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])

# Exploration Resources

Now that you know the basics, you can explore thousands of models on the Hugging Face Hub:

- [Transformers Pipeline Documentation](https://huggingface.co/docs/transformers/main_classes/pipelines)
- [Hugging Face Model Hub](https://huggingface.co/models)

You can use almost any model you find there by passing its name into the `model=` parameter of the `pipeline()` function.